# 01 — Policy Evaluation
**Week 4 | Dynamic Programming**

**Policy Evaluation** answers: *How good is a given policy π?*

We iteratively apply the Bellman expectation equation until V converges:

$$V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V_k(s') \right]$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# ---- Reuse Grid World from Week 3 ----
class GridWorld:
    ACTIONS = {0:(-1,0), 1:(1,0), 2:(0,-1), 3:(0,1)}
    ACTION_SYMBOLS = {0:'↑', 1:'↓', 2:'←', 3:'→'}
    def __init__(self, size=5):
        self.size = size; self.start=(0,0); self.goal=(size-1,size-1)
        self.pits = {(1,1),(1,3),(3,1),(3,3)}
    def n_states(self): return self.size**2
    def n_actions(self): return 4
    def transitions(self, s, a):
        """Return list of (prob, next_s, reward, done)."""
        r, c = divmod(s, self.size)
        dr, dc = self.ACTIONS[a]
        nr = max(0, min(self.size-1, r+dr)); nc = max(0, min(self.size-1, c+dc))
        ns = nr*self.size + nc
        if (nr,nc) == self.goal:       return [(1.0, ns, +10.0, True)]
        if (nr,nc) in self.pits:       return [(1.0, ns,  -5.0, True)]
        return [(1.0, ns, -0.1, False)]

env = GridWorld()

In [ ]:
def policy_evaluation(env, policy, gamma=0.99, theta=1e-6, max_iter=1000):
    """
    Iterative policy evaluation.
    policy: array of shape (n_states, n_actions) — action probabilities.
    Returns V: array of shape (n_states,)
    """
    V = np.zeros(env.n_states())
    history = [V.copy()]
    for iteration in range(max_iter):
        delta = 0.0
        V_new = np.zeros_like(V)
        for s in range(env.n_states()):
            v = 0.0
            for a in range(env.n_actions()):
                for prob, ns, reward, done in env.transitions(s, a):
                    v += policy[s, a] * prob * (reward + (0 if done else gamma * V[s]))
            V_new[s] = v
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        history.append(V.copy())
        if delta < theta:
            print(f"Converged in {iteration+1} iterations (Δ={delta:.2e})")
            break
    return V, history

In [ ]:
# Uniform random policy
n_s, n_a = env.n_states(), env.n_actions()
uniform_policy = np.ones((n_s, n_a)) / n_a

V, history = policy_evaluation(env, uniform_policy)

In [ ]:
# Visualise final value function
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Heatmap
im = axes[0].imshow(V.reshape(5,5), cmap='RdYlGn')
plt.colorbar(im, ax=axes[0])
for r in range(5):
    for c in range(5):
        label = ''
        if (r,c)==env.goal: label='G'
        elif (r,c)==env.start: label='S'
        elif (r,c) in env.pits: label='✕'
        axes[0].text(c, r, f'{label}\n{V[r*5+c]:.1f}', ha='center', va='center', fontsize=9)
axes[0].set_title('V^π (uniform random policy)'); axes[0].set_xticks([]); axes[0].set_yticks([])

# Convergence curve
max_delta = [np.max(np.abs(history[i+1]-history[i])) for i in range(len(history)-1)]
axes[1].semilogy(max_delta, color='steelblue', linewidth=2)
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Max |ΔV|')
axes[1].set_title('Convergence of Policy Evaluation')
plt.tight_layout(); plt.show()

## ✅ Exercises
1. Try γ=0.5, γ=0.9, γ=0.999. How does the value function shape change?
2. Create a policy that always moves RIGHT. Evaluate it. What happens to states on the rightmost column?
3. **Challenge**: implement **in-place** policy evaluation (update V[s] immediately instead of V_new). Does it converge faster?

In [ ]:
1) V05, _ = policy_evaluation(env, uniform_policy, gamma=0.5)
V09, _ = policy_evaluation(env, uniform_policy, gamma=0.9)
V0999, _ = policy_evaluation(env, uniform_policy, gamma=0.999)

print(V05.reshape(5,5))
print(V09.reshape(5,5))
print(V0999.reshape(5,5))

As γ increases, future rewards become more important. With γ=0.5, values are concentrated near the goal because distant rewards are heavily discounted. With γ=0.9, the influence of the goal spreads further across the grid. With γ=0.999, almost all states receive higher values because future rewards are considered nearly as important as immediate rewards.

In [ ]:
2) right_policy = np.zeros((n_s, n_a))
right_policy[:, 3] = 1.0   # action 3 = RIGHT

V_right, _ = policy_evaluation(env, right_policy)

print(V_right.reshape(5,5))

States in the rightmost column cannot move further right. Therefore, the agent repeatedly stays in the same column or position, causing the value estimates there to stabilize. States that cannot eventually reach the goal have low values, while states with a path to the goal retain higher values.

In [ ]:
3) def policy_evaluation_inplace(env, policy, gamma=0.99, theta=1e-6, max_iter=1000):
    V = np.zeros(env.n_states())

    for iteration in range(max_iter):
        delta = 0

        for s in range(env.n_states()):
            old_v = V[s]

            v = 0
            for a in range(env.n_actions()):
                for prob, ns, reward, done in env.transitions(s, a):
                    v += policy[s, a] * prob * (
                        reward + gamma * (0 if done else V[ns])
                    )

            V[s] = v
            delta = max(delta, abs(old_v - v))

        if delta < theta:
            print("Converged after", iteration + 1, "iterations")
            break

    return V

V_inplace = policy_evaluation_inplace(env, uniform_policy)

The in-place version updates values immediately and uses the newest estimates during the same iteration. Because information propagates more quickly through the state space, it usually converges in fewer iterations than the version that computes an entirely new value table before updating.